### Imports

In [1]:
import os
import re
import json
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import pdfplumber
import random
from docx import Document
from collections import Counter

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("BAAI/bge-m3")
# 10% overlap as specified in class.
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=500, 
    chunk_overlap=50)

### Gutenberg Novels

In [3]:
os.chdir('../Corpus/Gutenberg')

In [4]:
# Getting the files.
gutenberg_nov = [book for book in os.listdir() if book.endswith('.txt')]

In [5]:
# Making a place where chunks and books flagged for inconsistent gutenberg boilerplates go.
all_chunks = []
flagged_books = []

In [6]:
# Making a metadata pulling function. I named the files to make this easier.
def pull_meta(file, type):
    name = os.path.splitext(file)[0]
    author_part, title_parts = name.split('_', 1)
    title = title_parts.replace('_', ' ').title()
    author = " and ".join(a.capitalize() for a in author_part.split('-'))
    
    return {
        'source': name,
        'author': author,
        'title': title,
        'type': type
    }

In [ ]:
# Checking gutenberg files for consistency to ensure consistent chunking.
for book in gutenberg_nov:
    with open(book, 'r', encoding='utf-8') as b:
        text = b.read()
    
    if not re.search(r"START OF (THE|THIS) PROJECT GUTENBERG", text):
        flagged_books.append(book)
        continue
    if not re.search(r"END OF (THE|THIS) PROJECT GUTENBERG", text):
        flagged_books.append(book)
        continue

    # Cleaning before chunking.
    start = re.search(r"START OF (THE|THIS) PROJECT GUTENBERG", text).end()
    end = re.search(r"END OF (THE|THIS) PROJECT GUTENBERG", text).start()
    text = text[start:end]

    # Chunking
    chunks = text_splitter.create_documents([text])

    # Adding metadata
    metadata = pull_meta(book, type='novel')
    for chunk in chunks:
        chunk.metadata.update(metadata)
    all_chunks.extend(chunks)

if flagged_books:
    print("books flagged. Check files.")
else:
    print("No books flagged.")

No books flagged.


### Rulebooks

In [8]:
os.chdir('../Rulebooks')

In [9]:
# If the rulebook file is empty, it is probably a scanned PDF.
flagged_rulebooks = []

# Pulling rulebook files.
rulebook_files = [rule for rule in os.listdir() if rule.endswith('.pdf')]

for rulebook in rulebook_files:
    with pdfplumber.open(rulebook) as pdf:
        text = ''
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

    if not text.strip():
        flagged_rulebooks.append(rulebook)
        continue

    # chunking rulebook files.
    chunks = text_splitter.create_documents([text])
    metadata = pull_meta(rulebook, type='rulebook')
    for chunk in chunks:
        chunk.metadata.update(metadata)
    all_chunks.extend(chunks)

if flagged_rulebooks:
    print("rulebooks flagged. Check files.")
else:
    print("No rulebooks flagged.")

No rulebooks flagged.


In [10]:
# For extracting JSONs.
def json_puller(file, key_dict):
    texts = []
    if isinstance(file, dict):
        for k, v in file.items():
            if k in key_dict and isinstance(v, str):
                texts.append(v)
            # Take lists, but don't append a list of dictionaries.
            elif k in key_dict and isinstance(v, list) and all(isinstance(i, str) for i in v):
                texts.extend(v)
            else:
                texts.extend(json_puller(v, key_dict))
    elif isinstance(file, list):
        for i in file:
            texts.extend(json_puller(i, key_dict))
    return texts

In [11]:
# Chunk JSON Files.
json_files = [file for file in os.listdir() if file.endswith('.json')]
json_keys = {
    'tomkin_ironsworn_starforged.json': {'description', 'summary', 'quest_starter', 'your_character',
                                         'text', 'name'}
}

for file in json_files:
    with open(file, 'r', encoding='utf-8') as j:
        json_dat = json.load(j)
    text = '\n\n'.join(json_puller(json_dat, json_keys[file]))
    
    chunks = text_splitter.create_documents([text])
    # Only one JSON, or I would need a more robust solution.
    metadata = pull_meta(file, type='rulebook')
    for chunk in chunks:
        chunk.metadata.update(metadata)

    all_chunks.extend(chunks)

In [12]:
# For extracting and chunking Markdown File:
md_file = [file for file in os.listdir() if file.endswith('.md')]

for file in md_file:
    with open(file, 'r', encoding='utf-8') as m:
        text = m.read()
    
    chunks = text_splitter.create_documents([text])
    metadata = pull_meta(file, type='rulebook')
    for chunk in chunks:
        chunk.metadata.update(metadata)
    all_chunks.extend(chunks)

In [13]:
# For extracting and chunking Docx File:
docx_file = [file for file in os.listdir() if file.endswith('.docx')]

for file in docx_file:
    doc = Document(file)
    text = '\n\n'.join([parag.text for parag in doc.paragraphs])
    
    chunks = text_splitter.create_documents([text])
    metadata = pull_meta(file, type='rulebook')
    for chunk in chunks:
        chunk.metadata.update(metadata)
    all_chunks.extend(chunks)

### Scenarios

In [14]:
os.chdir('../Scenarios')

In [ ]:
# If the scenario file is empty, it is probably a scanned PDF.
flagged_scenarios = []

# Pulling scenario files.
scenario_files = [sc for sc in os.listdir() if sc.endswith('.pdf')]

for sc in scenario_files:
    with pdfplumber.open(sc) as pdf:
        text = ''
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

    if not text.strip():
        flagged_scenarios.append(sc)
        continue

    # chunking rulebook files.
    chunks = text_splitter.create_documents([text])
    metadata = pull_meta(sc, type='scenario')
    for chunk in chunks:
        chunk.metadata.update(metadata)
    all_chunks.extend(chunks)

if flagged_scenarios:
    print("scenarios flagged. Check files.")
else:
    print("No scenarios flagged.")

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

No scenarios flagged.


### Check Chunks

In [16]:
# Number of chunks:
print(f'total chunks: {len(all_chunks)}')

# By type:
type_count = Counter(c.metadata['type'] for c in all_chunks)
for t, count in type_count.items():
    print(f'{t}: {count} chunks')

total chunks: 15872
novel: 12209 chunks
rulebook: 3001 chunks
scenario: 662 chunks


In [19]:
# Chunks per source specifically
source_chunks = Counter(c.metadata['source'] for c in all_chunks)
for s, count in source_chunks.items():
    print(f'{s}: {count} chunks')

anonymouos_the_poetic_edda: 604 chunks
anonymous_beowulf: 173 chunks
anonymous_the_arabian_nights_entertainment: 363 chunks
defoe_the_life_and_adventures_of_robinson_crusoe: 441 chunks
dickens_great_expectations: 644 chunks
dostoyevsky_crime_and_punishment: 769 chunks
doyle_the_adventures_of_sherlock_holmes: 357 chunks
doyle_the_hound_of_baskervilles: 195 chunks
dumas_the_Three_musketeers: 789 chunks
dunsay_the_king_of_elflands_daughter: 249 chunks
eddison_the_worm_ouroboros_a_romance: 656 chunks
emily_bronte_wuthering_heights: 444 chunks
grey_riders_of_the_purple_sage: 393 chunks
grimm_grimms_fairy_tales: 371 chunks
hodgeson_the_night_land: 608 chunks
homer_the_odyssey: 458 chunks
jerome_three_men_in_a_boat: 220 chunks
lovecraft_at_the_mountains_of_madness: 136 chunks
machen_the_great_god_pan: 80 chunks
malory_king_arthur_and_the_knights_of_the_round_table: 351 chunks
milne_winnie_the_pooh: 85 chunks
morris_the_well_at_the_worlds_end: 797 chunks
nietzsche_beyond_good_and_evil: 272 chu

### Export Chunks

In [ ]:
os.chdir('../')

In [24]:
chunks_export = [
    {
        'text': chunk.page_content,
        'metadata': chunk.metadata
    }
    for chunk in all_chunks
]

with open('corpus_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_export, f, indent=2)

print(f'total chunks: {len(all_chunks)}')

total chunks: 15872


Citation:

Brainstorming and refinement: 

Anthropic. (2026). Claude Sonnet 4.6 [AI language model]. https://claude.ai 